In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import tqdm
import cobra
from cobra.io.mat import load_matlab_model

from util import update_reaction_bounds

import warnings
warnings.filterwarnings("ignore")


conf = cobra.Configuration()
conf.solver = "glpk"
scaler = MinMaxScaler()

In [19]:
default_model = load_matlab_model("model/Human-GEM.mat")
default_model

Name,ihuman
Memory address,2242950cc20
Number of metabolites,8369
Number of reactions,13070
Number of genes,3067
Number of groups,0
Objective expression,1.0*MAR13082 - 1.0*MAR13082_reverse_11d67
Compartments,"Cytosol, Extracellular, Lysosome, Endoplasmic reticulum, Mitochondria, Peroxisome, Golgi apparatus, Nucleus, Inner mitochondria"


In [20]:
gene_expr_df = pd.read_csv("data/normalised_cpm_counts.csv", index_col=0)
gene_expr_df.head()

,Sample_1,Sample_2,Sample_3
ENSG00000000003,7.263192,5.839909,14.434692
ENSG00000000419,3.034660,3.542537,8.574593
ENSG00000000457,5.030148,6.022994,3.997502
ENSG00000000460,2.014106,1.241415,3.733868
ENSG00000001036,5.001140,4.396138,6.089214


### Select metabolic genes

In [21]:
genes_in_model = [gene.id for gene in default_model.genes]
gene_expr_df = gene_expr_df[gene_expr_df.index.isin(genes_in_model)]
print(gene_expr_df.index.isin(genes_in_model).sum())
gene_expr_df.head()

1914


,Sample_1,Sample_2,Sample_3
ENSG00000000419,3.034660,3.542537,8.574593
ENSG00000001036,5.001140,4.396138,6.089214
ENSG00000001084,8.783728,9.222366,11.402419
ENSG00000002549,5.228614,1.481012,12.804489
ENSG00000002587,4.500259,4.998904,1.144002


### Normalise metabolic genes

In [22]:
gene_expr_df = pd.DataFrame(scaler.fit_transform(gene_expr_df), index=gene_expr_df.index, columns=gene_expr_df.columns)
gene_expr_df

,Sample_1,Sample_2,Sample_3
ENSG00000000419,0.001768,0.001814,0.007848
ENSG00000001036,0.002957,0.002264,0.005545
ENSG00000001084,0.005246,0.004813,0.010468
ENSG00000002549,0.003095,0.000725,0.011768
ENSG00000002587,0.002654,0.002583,0.000962
...,...,...,...
ENSG00000276043,0.001782,0.000299,0.012344
ENSG00000276293,0.006541,0.006836,0.006343
ENSG00000278540,0.008684,0.005791,0.009540
ENSG00000278619,0.001246,0.001009,0.002242


### Metabolic metadata

In [23]:
rxn_data = []
for rxn in default_model.reactions:
    rxn_data.append({
        "ID": rxn.id,
        "Name": rxn.name,
        "GPR": str(rxn.gpr),
        "Lower_Bound": rxn.lower_bound,
        "Upper_Bound": rxn.upper_bound
    })

rxn_df = pd.DataFrame(rxn_data)
rxn_df[rxn_df['GPR'] != '']

,ID,Name,GPR,Lower_Bound,Upper_Bound
0,MAR03905,ethanol:NAD+ oxidoreductase,ENSG00000147576 or ENSG00000172955 or ENSG0000...,0.0,1000.0
1,MAR03907,Ethanol:NADP+ oxidoreductase,ENSG00000117448,0.0,1000.0
2,MAR04097,Acetate:CoA ligase (AMP-forming),ENSG00000131069,0.0,1000.0
3,MAR04099,Acetate:CoA ligase (AMP-forming),ENSG00000111058 or ENSG00000154930,0.0,1000.0
4,MAR04108,acetyl adenylate:CoA acetyltransferase,ENSG00000131069,0.0,1000.0
...,...,...,...,...,...
13058,MAR10125,,ENSG00000073734 or ENSG00000108846,0.0,1000.0
13059,MAR10126,,ENSG00000073734 or ENSG00000108846,0.0,1000.0
13060,MAR10127,,ENSG00000100652,0.0,1000.0
13066,MAR13083,,ENSG00000010932,0.0,1000.0


### Perform FBA for each sample

In [24]:
FBA_Fluxes = {}

for sample in tqdm.tqdm(range(gene_expr_df.shape[1])):
    model =  update_reaction_bounds(default_model, gene_expr_df.iloc[:, sample], gamma=2)
    solution = model.optimize()
    fluxes = pd.DataFrame(solution.fluxes)
    model_name = gene_expr_df.columns[sample]

    FBA_Fluxes[gene_expr_df.columns[sample]] = solution.fluxes

FBA_Fluxes = pd.DataFrame(FBA_Fluxes)
FBA_Fluxes

  0%|          | 0/3 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:19<00:00,  6.49s/it]


,Sample_1,Sample_2,Sample_3
MAR03905,0.000000,0.000000,0.000000
MAR03907,0.000000,0.000000,12.743355
MAR04097,0.000000,0.000000,0.000000
MAR04099,0.000000,0.000000,0.000000
MAR04108,0.000000,0.000000,0.000000
...,...,...,...
MAR10132,0.000000,0.000000,0.000000
MAR13083,0.000000,0.000000,0.000000
MAR13084,0.000000,0.000000,0.000000
MAR13085,0.000000,0.000000,0.000000


In [25]:
FBA_Fluxes.to_csv("results/FBA_Fluxes.csv")